# Building Decision Tree From Scratch

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import datetime as dt
import seaborn as sns

from ml_world_cup_predictor.feature_engineering import get_data_for_tree_build
from ml_world_cup_predictor.plot_theme import apply_theme, set_titles
from ml_world_cup_predictor.config import WC_START

apply_theme()



# Loading and Build Working Dataset for Decision Tree
final_df = get_data_for_tree_build(['elo_difference','form_difference'],'result')
final_df = final_df.dropna()



48852
48852


In [ ]:
final_df.tail() 

,home_elo,away_elo,home_form,away_form,elo_difference,form_difference,result
48847,2336.832602,2450.376945,42,50,-113.544343,-8,L
48848,2364.596162,2566.509969,44,45,-201.913807,-1,L
48849,2172.861566,2248.877303,40,39,-76.015737,1,L
48850,2493.240493,2081.458269,52,33,411.782223,19,W
48851,2342.057332,2331.779031,40,35,10.278301,5,W


In [ ]:
## Building entropy Score Function
def entropy(label_counts,total):

    # Not possible to calculate entropy on non-existent data
    if total == 0:
        return 0
    
    # Initialise Entropy Value
    entropy_value = 0

    # Iterate the class count
    for label in label_counts:
        if label == 0:
            continue
        
        probability = label/total
        entropy_value -= probability * np.log2(probability)
    
    return entropy_value


def find_best_threshold(feature,y):

    # Paring up each feature value with corresponding label and sorting by value
    paired = sorted(zip(feature,y))

    # Full Length of the data
    n = len(y)

    # The counts when the filter is at the furthest left of the data
    labels, counts = np.unique(y,return_counts=True)
    right_count = dict(zip(labels,counts))
    left_count = {label: 0 for label in labels}

    # Initialised before the for loop
    best_gain = -1
    best_threshold = None


    parent_entropy = entropy(right_count.values(),n)

    # Iterate over the index of the data
    for i in range(n-1):

        # Get the current data and label
        value_i, label_i = paired[i]

        # Increment the left count of whatever label is at this index as we move the parition to the right of it
        left_count[label_i] += 1
        # Reduce the right count at this index as this label goes to the lift of the partition 
        right_count[label_i] -= 1

        # Identify the next value and label
        value_next,label_next = paired[i+1]

        # If there is no change in class continue because .....
        if label_i == label_next:
            continue
        
        # Calculate the total size of both sides
        n_left = i + 1
        n_right = n - n_left

        # Calculate the entropy for each child node
        left_entropy = entropy(left_count.values(),n_left)
        right_entropy = entropy(right_count.values(),n_right)

        information_gain = parent_entropy - (n_left/n)*left_entropy - (n_right/n)*right_entropy

        if information_gain > best_gain:
            best_gain = information_gain
            
            best_threshold = (value_i + value_next)/2
        
    return best_threshold,best_gain
